# AgentSZZ Debug

Where the agent succeeds, where it fails, and whether its failures are a matter of capability or of output format. The run is a repeated evaluation of `attribution-agent-szz-evaluation`, several runs per case, scored against the defect commits recorded in the validation context index.

A prediction counts as correct when it prefixes one of the recorded defect commits, or is prefixed by one, from seven characters upward. Each trace falls into exactly one of four outcomes.

| Outcome | Meaning |
|---|---|
| `correct` | the predicted commit is one of the recorded defects |
| `incorrect` | a commit was predicted and it is not one of them |
| `empty` | no commit reached the output field |
| `error` | the graph itself failed, for instance on a recursion limit |

Cases carry the category of their defect within the knowledge graph the agent traverses. The category says how far the defect sits from the correction.

| Category | Meaning |
|---|---|
| `blame` | the defect is the commit that git blame reaches directly |
| `blame-ancestor` | the defect lies above the blame result in its history |
| `correction-ancestor` | the defect lies in the ancestry of the correction |
| `outside` | the defect appears in no commit of the index at all |

## Inputs

The evaluation traces hold one event stream per run. The context index holds the recorded defect commits and the commit pool of each case, from which the categories are taken.

In [1]:
import io
import json
import re
from pathlib import Path

import polars as pl

from benchmark.attribution.evaluation import (
    MessageProcessor,
    TokenProcessor,
    TraceSerializer,
)
from benchmark.tracing import Trace
from benchmark_notebook.io import CloudStorage

In [2]:
traces = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        key="attribution-agent-szz-evaluation",
        version="1.0.1",
        location="main",
        partition="default",
    )
)

index = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        key="ingest-validation-context-index",
        version="2.0.0",
        location="main",
        partition="default",
    )
)

## Predictions

The root LangGraph span carries the outcome. A completed run reports the predicted commit under `node_end`, a failed one reports the exception under `node_error`. Where a trace holds both, the completed event wins.

In [3]:
def _extract_introducer(payload: str) -> str | None:
    p = json.loads(payload)
    return p.get("outputs", {}).get("introducer")


def _extract_error(payload: str) -> str | None:
    p = json.loads(payload)
    return p.get("error")


# Root LangGraph node_end events carry the prediction
predictions_end = (
    traces.filter(
        (pl.col("node") == "LangGraph")
        & (pl.col("event") == "node_end")
        & pl.col("parent").is_null()
    )
    .with_columns(
        pl.col("payload")
        .map_elements(_extract_introducer, return_dtype=pl.String)
        .alias("predicted"),
    )
    .select(["trace", "repository", "correction", "predicted"])
    .with_columns(pl.lit(None).cast(pl.String).alias("error"))
)

# Root LangGraph node_error events -- graph crashed (e.g. recursion limit)
predictions_error = (
    traces.filter(
        (pl.col("node") == "LangGraph")
        & (pl.col("event") == "node_error")
        & pl.col("parent").is_null()
    )
    .with_columns(
        pl.lit(None).cast(pl.String).alias("predicted"),
        pl.col("payload")
        .map_elements(_extract_error, return_dtype=pl.String)
        .alias("error"),
    )
    .select(["trace", "repository", "correction", "predicted", "error"])
)

predictions = pl.concat([predictions_end, predictions_error])

# Deduplicate: keep node_end over node_error when both exist for the same trace
predictions = predictions.sort("error", nulls_last=False).group_by("trace").first()
predictions.shape

(2268, 5)

## Outcome and category

Predictions are matched against the recorded defects and joined to the case category. Traces without a category carry a defect the index never saw.

In [4]:
def _match(predicted: str | None, defects: list[str]) -> bool:
    if predicted is None:
        return False
    p = predicted.lower().strip()
    if len(p) < 7:
        return False
    return any(d.lower().startswith(p) or p.startswith(d.lower()) for d in defects)


ground = index.select(["repository", "correction", "defects"])
joined = predictions.join(ground, on=["repository", "correction"], how="left")

joined = joined.with_columns(
    pl.cum_count("trace").over(["repository", "correction"]).alias("run")
)

evaluated = joined.with_columns(
    pl.struct(["predicted", "defects"])
    .map_elements(
        lambda row: _match(row["predicted"], row["defects"] or []),
        return_dtype=pl.Boolean,
    )
    .alias("correct"),
    (
        pl.col("predicted").is_not_null() & (pl.col("predicted").str.len_chars() >= 7)
    ).alias("has_prediction"),
)

# -- Outcome: correct / incorrect / error / empty (mutually exclusive) --
evaluated = evaluated.with_columns(
    pl.when(pl.col("error").is_not_null())
    .then(pl.lit("error"))
    .when(pl.col("correct"))
    .then(pl.lit("correct"))
    .when(pl.col("has_prediction"))
    .then(pl.lit("incorrect"))
    .otherwise(pl.lit("empty"))
    .alias("outcome")
)

# -- Join category from commits (the category of the ground-truth defect) --
defect_categories = (
    index.select(["repository", "correction", "defects", "commits"])
    .explode("defects")
    .explode("commits")
    .drop_nulls("commits")
    .unnest("commits")
    .filter(pl.col("commit") == pl.col("defects"))
    .group_by(["repository", "correction"])
    .agg(pl.col("category").first())
    .with_columns(pl.col("category").str.replace_all("_", "-"))
)

# A defect that appears in no commit of the index has no category of its own and is
# booked as outside, meaning the ground truth lies beyond the knowledge graph the
# agent can reach.
evaluated = evaluated.join(
    defect_categories, on=["repository", "correction"], how="left"
).with_columns(pl.col("category").fill_null("outside"))
evaluated.group_by("outcome").len().sort("outcome")

outcome,len
str,u32
"""correct""",1617
"""empty""",93
"""error""",39
"""incorrect""",519


## Case breakdown

The share of each outcome, overall and split by how far the defect sits from the correction.

In [5]:
def _pct(n, total):
    pct = 100 * n / total if total > 0 else 0.0
    return f"{pct:.1f}% ({n})"


def _breakdown(frame, group_col=None):
    if group_col:
        raw = (
            frame.group_by(group_col)
            .agg(
                pl.len().alias("total"),
                pl.col("correct").sum().alias("correct"),
                (pl.col("has_prediction") & ~pl.col("correct"))
                .sum()
                .alias("incorrect"),
                (~pl.col("has_prediction")).sum().alias("no_answer"),
                pl.col("error").is_not_null().sum().alias("errors"),
            )
            .sort(group_col)
        )
    else:
        raw = frame.select(
            pl.len().alias("total"),
            pl.col("correct").sum().alias("correct"),
            (pl.col("has_prediction") & ~pl.col("correct")).sum().alias("incorrect"),
            (~pl.col("has_prediction")).sum().alias("no_answer"),
            pl.col("error").is_not_null().sum().alias("errors"),
        )

    rows = []
    for r in raw.iter_rows(named=True):
        t = r["total"]
        row = {
            "total": t,
            "correct": _pct(r["correct"], t),
            "incorrect": _pct(r["incorrect"], t),
            "no_answer": _pct(r["no_answer"], t),
            "errors": _pct(r["errors"], t),
        }
        if group_col:
            row[group_col] = r[group_col]
        rows.append(row)

    cols = [group_col] if group_col else []
    cols += ["total", "correct", "incorrect", "no_answer", "errors"]
    return pl.DataFrame(rows).select(cols)


print("=== Overall ===")
display(_breakdown(evaluated))

print("\n=== By Category ===")
display(_breakdown(evaluated, "category"))

=== Overall ===


total,correct,incorrect,no_answer,errors
i64,str,str,str,str
2268,"""71.3% (1617)""","""22.9% (519)""","""5.8% (132)""","""1.7% (39)"""



=== By Category ===


category,total,correct,incorrect,no_answer,errors
str,i64,str,str,str,str
"""blame""",1728,"""83.2% (1437)""","""12.3% (212)""","""4.6% (79)""","""1.3% (23)"""
"""blame-ancestor""",72,"""15.3% (11)""","""72.2% (52)""","""12.5% (9)""","""4.2% (3)"""
"""correction-ancestor""",333,"""41.4% (138)""","""48.6% (162)""","""9.9% (33)""","""3.0% (10)"""
"""outside""",135,"""23.0% (31)""","""68.9% (93)""","""8.1% (11)""","""2.2% (3)"""


## Where the wrong answers land

An incorrect prediction still points somewhere. Matching it back against the commit pool of the same case shows which part of the graph the agent reached instead. This is the category of the **predicted** commit, not of the ground truth, so the two form a confusion matrix over the graph.

In [6]:
# Long table of (repository, correction, commit, category) for every commit in the index
commits_long = (
    index.select(["repository", "correction", "commits"])
    .explode("commits")
    .drop_nulls("commits")
    .unnest("commits")
    .select(["repository", "correction", "commit", "category"])
    .with_columns(pl.col("category").str.replace_all("_", "-"))
)

incorrect = (
    evaluated.filter(pl.col("outcome") == "incorrect")
    .select(["trace", "repository", "correction", "predicted", "category"])
    .rename({"category": "truth_category"})
)

# Match predicted SHA to a commit in the same case via prefix on either side
incorrect_pool = incorrect.join(
    commits_long, on=["repository", "correction"], how="left"
).with_columns(
    pl.col("predicted").str.to_lowercase().alias("_p"),
    pl.col("commit").str.to_lowercase().alias("_c"),
)
incorrect_matched = (
    incorrect_pool.filter(
        pl.col("_c").str.starts_with(pl.col("_p"))
        | pl.col("_p").str.starts_with(pl.col("_c"))
    )
    .group_by("trace")
    .agg(pl.col("category").first().alias("predicted_category"))
)

incorrect_classified = incorrect.join(
    incorrect_matched, on="trace", how="left"
).with_columns(pl.col("predicted_category").fill_null("unknown"))

print(f"=== Incorrect predictions: {incorrect_classified.height} ===")

print("\n=== Distribution by predicted category ===")
display(
    incorrect_classified.group_by("predicted_category")
    .agg(pl.len().alias("n"))
    .with_columns(
        (pl.col("n") / pl.lit(incorrect_classified.height) * 100)
        .round(1)
        .alias("pct")
    )
    .sort("n", descending=True)
)

print("\n=== Predicted category x truth category ===")
display(
    incorrect_classified.group_by(["truth_category", "predicted_category"])
    .agg(pl.len().alias("n"))
    .sort(["truth_category", "predicted_category"])
    .pivot(on="predicted_category", index="truth_category", values="n")
    .fill_null(0)
    .sort("truth_category")
)

=== Incorrect predictions: 519 ===

=== Distribution by predicted category ===


predicted_category,n,pct
str,u32,f64
"""blame""",377,72.6
"""blame-ancestor""",69,13.3
"""unknown""",58,11.2
"""correction""",10,1.9
"""correction-ancestor""",5,1.0



=== Predicted category x truth category ===


truth_category,blame,blame-ancestor,correction,correction-ancestor,unknown
str,u32,u32,u32,u32,u32
"""blame""",123,39,6,2,42
"""blame-ancestor""",41,10,1,0,0
"""correction-ancestor""",138,12,3,1,8
"""outside""",75,8,0,2,8


## Cases without a prediction

Traces whose output field holds no usable commit. The export below writes one OTLP file per trace for inspection in a viewer. It produces no result, it is a tool.

In [7]:
no_answer_cases = evaluated.filter(~pl.col("has_prediction"))
no_answer_keys = no_answer_cases.select(
    ["trace", "repository", "correction", "run", "error", "defects"]
).unique()

print(f"{no_answer_keys.height} traces without a parseable prediction.")
no_answer_keys

132 traces without a parseable prediction.


trace,repository,correction,run,error,defects
str,str,str,u32,str,list[str]
"""451c868c2d3f4d8ba5768079f5595e…","""sclabs/projectile""","""7edc7fd44310fafa79a30660b33eb3…",6,null,"[""1fc5c0a5d868763de36d47d98891f5d2da2a5aa0""]"
"""81c831fc561548d98ac93365b96b36…","""obspy/obspy""","""968ff86ea1f297a32e454962f2ac8b…",6,null,"[""6a914e66d02ccd85f68dce2d4956b666459a633b""]"
"""ac7c5bf3d0864992bbf9b066636301…","""nouiz/Theano""","""d714304c311d8e9bf9b88eac92a89d…",8,null,"[""41a8e89b63f5b4d4f9a0b942554ed5483e1ceab5""]"
"""1b8191af36ce4a0eb54bacda4fb80b…","""janelia-flyem/DVIDSparkService…","""ceb21abd16984e2aeccaad6b382445…",2,"""Error code: 400 - {'id': 'ogwS…","[""33aabcbce6be10d614b272fc15d16f458e219663""]"
"""0889c191eb414244ac9ccabd527ddf…","""terminalmage/salt""","""c8a5decdf10a6be46631b0b92742c7…",8,null,"[""17b4538567de730ebd163f706c3880cbe4251715""]"
…,…,…,…,…,…
"""178a01ee43c6427a90ef0ae69edd2c…","""trezor/trezor-firmware""","""34ec1ed294edb2b2245c3baf14a08b…",6,"""Could not parse response conte…","[""bb2556a22cd3ffdc612a06da7d40eeb36a38de10""]"
"""08e48d0a021f44b5be3f28c933e82c…","""f-droid/fdroidserver""","""e75cabfe77e6823b99d9b9df389930…",2,"""Could not parse response conte…","[""04db6870d17852026ff00120814988af904370db""]"
"""7e5cfd3fd8b0480c93883be0ec30cd…","""praet/sluk""","""0bd44a41672fc47047913c0a0b0c8a…",5,null,"[""537dc59c7c922a0b7f4d1a2732d171076dfcf05f""]"


In [8]:
output_dir = Path(".traces") / "no-answer"

otlp_paths = []

for row in no_answer_keys.iter_rows(named=True):
    trace_id = row["trace"]
    repo = row["repository"]
    correction = row["correction"]

    trace_events = traces.filter(pl.col("trace") == trace_id).sort("sequence")
    trace_obj = Trace(
        seed=trace_id,
        metadata={
            "repository": repo,
            "correction": correction,
        },
    )
    for event in trace_events.iter_rows(named=True):
        trace_obj._events.append(
            {
                "trace": event["trace"],
                "sequence": event["sequence"],
                "timestamp": event["timestamp"],
                "node": event["node"],
                "span": event["span"],
                "parent": event["parent"],
                "event": event["event"],
                "payload": event["payload"],
            }
        )
    trace_obj._sequence = trace_events.height

    dest = output_dir / repo / f"{correction[:12]}.otlp.json"
    dest.parent.mkdir(parents=True, exist_ok=True)

    serializer = TraceSerializer(
        processors=[
            MessageProcessor(),
            TokenProcessor(),
        ]
    )
    resource = {
        "service.name": "AgentSZZ",
        "repository": repo,
        "correction": correction,
    }
    buf = io.StringIO()
    serializer.serialize(trace_obj, buf, resource=resource)
    dest.write_text(buf.getvalue(), encoding="utf-8")
    otlp_paths.append(
        {
            "repository": repo,
            "correction": correction,
            "path": str(dest),
            "events": trace_events.height,
        }
    )

otlp_index = pl.DataFrame(otlp_paths)
otlp_index

repository,correction,path,events
str,str,str,i64
"""sclabs/projectile""","""7edc7fd44310fafa79a30660b33eb3…",""".traces/no-answer/sclabs/proje…",188
"""obspy/obspy""","""968ff86ea1f297a32e454962f2ac8b…",""".traces/no-answer/obspy/obspy/…",210
"""nouiz/Theano""","""d714304c311d8e9bf9b88eac92a89d…",""".traces/no-answer/nouiz/Theano…",400
"""janelia-flyem/DVIDSparkService…","""ceb21abd16984e2aeccaad6b382445…",""".traces/no-answer/janelia-flye…",378
"""terminalmage/salt""","""c8a5decdf10a6be46631b0b92742c7…",""".traces/no-answer/terminalmage…",392
…,…,…,…
"""trezor/trezor-firmware""","""34ec1ed294edb2b2245c3baf14a08b…",""".traces/no-answer/trezor/trezo…",294
"""f-droid/fdroidserver""","""e75cabfe77e6823b99d9b9df389930…",""".traces/no-answer/f-droid/fdro…",126
"""praet/sluk""","""0bd44a41672fc47047913c0a0b0c8a…",""".traces/no-answer/praet/sluk/0…",414


## Predictions the parser missed

A trace can carry no commit in its output field while the model names one in prose. The last generation of each such trace is scanned for hex strings of at least seven characters, and the root payload with it. A hit means the failure is one of format rather than of capability.

In [9]:
SHA_PATTERN = re.compile(r"\b[0-9a-f]{7,40}\b")

hidden = []

for row in no_answer_keys.iter_rows(named=True):
    trace_id = row["trace"]
    trace_events = traces.filter(pl.col("trace") == trace_id).sort("sequence")

    llm_ends = trace_events.filter(pl.col("event") == "llm_end")
    if llm_ends.height == 0:
        hidden.append(
            {
                "trace": trace_id,
                "repository": row["repository"],
                "correction": row["correction"],
                "error": row["error"],
                "defects": row["defects"],
                "last_generation": None,
                "sha_candidates": [],
                "has_hidden_sha": False,
                "form": "no_llm_output",
            }
        )
        continue

    last_llm = llm_ends.row(-1, named=True)
    payload = json.loads(last_llm["payload"])

    generation_text = ""
    for batch in payload.get("generations") or []:
        for g in batch if isinstance(batch, list) else [batch]:
            if not isinstance(g, dict):
                continue
            text = g.get("text") or ""
            generation_text += text
            msg = g.get("message") or {}
            if isinstance(msg, dict):
                content = msg.get("content") or ""
                if isinstance(content, str):
                    generation_text += " " + content
                for call in msg.get("tool_calls") or []:
                    args = json.dumps(call.get("args", {}))
                    generation_text += " " + args

    sha_matches = SHA_PATTERN.findall(generation_text.lower())

    # Also check the root node payload for partial outputs
    root_ends = trace_events.filter(
        (pl.col("node") == "LangGraph")
        & (pl.col("event").is_in(["node_end", "node_error"]))
        & pl.col("parent").is_null()
    )
    if root_ends.height > 0:
        root_payload = json.loads(root_ends.row(-1, named=True)["payload"])
        sha_matches += SHA_PATTERN.findall(json.dumps(root_payload).lower())

    sha_candidates = sorted(set(sha_matches))

    if row["error"]:
        form = "error"
    elif not generation_text.strip():
        form = "empty_generation"
    elif sha_candidates:
        form = "has_sha_in_text"
    else:
        form = "no_sha_at_all"

    snippet = generation_text[-500:] if len(generation_text) > 500 else generation_text

    hidden.append(
        {
            "trace": trace_id,
            "repository": row["repository"],
            "correction": row["correction"],
            "error": row["error"],
            "defects": row["defects"],
            "last_generation": snippet,
            "sha_candidates": sha_candidates,
            "has_hidden_sha": bool(sha_candidates),
            "form": form,
        }
    )

hidden_df = pl.DataFrame(hidden)
form_counts = (
    hidden_df.group_by("form")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
form_counts

form,count
str,u32
"""has_sha_in_text""",93
"""error""",38
"""no_llm_output""",1


In [10]:
with_sha = hidden_df.filter(pl.col("has_hidden_sha"))
with_sha.select(
    [
        "repository",
        "correction",
        "form",
        "sha_candidates",
        "defects",
        "last_generation",
    ]
)

repository,correction,form,sha_candidates,defects,last_generation
str,str,str,list[str],list[str],str
"""sclabs/projectile""","""7edc7fd44310fafa79a30660b33eb3…","""has_sha_in_text""","[""7edc7fd44310fafa79a30660b33eb30ef99c7bdd"", ""f60ca4d4807bd6d3ca110e93d800b912c29f0618""]","[""1fc5c0a5d868763de36d47d98891f5d2da2a5aa0""]","""{ ""introducer"": null }"""
"""obspy/obspy""","""968ff86ea1f297a32e454962f2ac8b…","""has_sha_in_text""","[""7769e96bb68e3e4e42d0bb563c9d0a2192d35e70"", ""948000000004"", ""968ff86ea1f297a32e454962f2ac8b19dcef6769""]","[""6a914e66d02ccd85f68dce2d4956b666459a633b""]","""{""introducer"": null}"""
"""nouiz/Theano""","""d714304c311d8e9bf9b88eac92a89d…","""has_sha_in_text""","[""7320e1b"", ""c42a18c6eb08b137a811036099dee9af805536f4"", ""d714304c311d8e9bf9b88eac92a89d04af99e470""]","[""41a8e89b63f5b4d4f9a0b942554ed5483e1ceab5""]","""{ ""introducer"": ""abergeron@…"
"""janelia-flyem/DVIDSparkService…","""ceb21abd16984e2aeccaad6b382445…","""error""","[""8815357cec7c2597e5b1f65f1a4f979a6b84d10c"", ""9f358dc77e584883""]","[""33aabcbce6be10d614b272fc15d16f458e219663""]",""" {""commit"": ""8815357cec7c2597…"
"""terminalmage/salt""","""c8a5decdf10a6be46631b0b92742c7…","""has_sha_in_text""","[""97699999998"", ""c7d3ac58efcda4178787cd81cfce9cabc1e797fe"", ""c8a5decdf10a6be46631b0b92742c73ab9eecb1f""]","[""17b4538567de730ebd163f706c3880cbe4251715""]","""{""introducer"": null}"""
…,…,…,…,…,…
"""trezor/trezor-firmware""","""34ec1ed294edb2b2245c3baf14a08b…","""error""","[""bb2556a22cd3ffdc612a06da7d40eeb36a38de10""]","[""bb2556a22cd3ffdc612a06da7d40eeb36a38de10""]",""" {""commit"": ""bb2556a22cd3ffdc…"
"""f-droid/fdroidserver""","""e75cabfe77e6823b99d9b9df389930…","""error""","[""04db6870d17852026ff00120814988af904370db"", ""e75cabfe77e6823b99d9b9df3899307fabc9d462""]","[""04db6870d17852026ff00120814988af904370db""]","""904370db`, when refactoring `s…"
"""praet/sluk""","""0bd44a41672fc47047913c0a0b0c8a…","""has_sha_in_text""","[""0bd44a41672fc47047913c0a0b0c8a6fb8448413"", ""bbaab23f33678f567b7f7c6df120072fd628cf08""]","[""537dc59c7c922a0b7f4d1a2732d171076dfcf05f""]","""{""introducer"": null}"""


## Headroom from the failure modes

The breakdown above counts every failure against the agent. The three levels here remove one failure mode at a time and bound what the agent would reach if the mode were fixed. The levels are nested, so accuracy rises by construction, and the size of the rise is the point.

| Level | Removes |
|---|---|
| Overall | nothing |
| Trace exists | runs whose graph crashed |
| Has SHA-like prediction | runs whose output field stayed empty |

In [11]:
def _accuracy(frame, label):
    n = frame.height
    correct = frame.filter(pl.col("correct")).height
    return {
        "level": label,
        "n": n,
        "correct": correct,
        "accuracy": correct / n if n > 0 else 0.0,
    }


def _stratified(frame, label_prefix=""):
    error_traces_local = hidden_df.filter(pl.col("form") == "error").select("trace")

    rows = []
    rows.append(_accuracy(frame, f"{label_prefix}Overall"))

    trace_ok = frame.join(error_traces_local, on="trace", how="anti")
    rows.append(_accuracy(trace_ok, f"{label_prefix}Trace exists (no error)"))

    has_sha = frame.filter(pl.col("has_prediction"))
    rows.append(_accuracy(has_sha, f"{label_prefix}Has SHA-like prediction"))
    return rows


# -- Overall --
print("=== Overall ===")
display(pl.DataFrame(_stratified(evaluated)))

# -- By Category --
print("\n=== By Category ===")
cat_rows = []
for cat in sorted(evaluated["category"].drop_nulls().unique().to_list()):
    cat_rows.extend(
        _stratified(evaluated.filter(pl.col("category") == cat), f"[{cat}] ")
    )
display(pl.DataFrame(cat_rows))

=== Overall ===


level,n,correct,accuracy
str,i64,i64,f64
"""Overall""",2268,1617,0.712963
"""Trace exists (no error)""",2230,1617,0.725112
"""Has SHA-like prediction""",2136,1617,0.757022



=== By Category ===


level,n,correct,accuracy
str,i64,i64,f64
"""[blame] Overall""",1728,1437,0.831597
"""[blame] Trace exists (no error…",1705,1437,0.842815
"""[blame] Has SHA-like predictio…",1649,1437,0.871437
"""[blame-ancestor] Overall""",72,11,0.152778
"""[blame-ancestor] Trace exists …",69,11,0.15942
…,…,…,…
"""[correction-ancestor] Trace ex…",324,138,0.425926
"""[correction-ancestor] Has SHA-…",300,138,0.46
"""[outside] Overall""",135,31,0.22963


## Pickaxe usage against outcome

How often the agent reached for the pickaxe tool, and whether reaching for it went with a better or a worse outcome.

In [12]:
# Count pickaxe tool_end events per trace
pickaxe_counts = (
    traces.filter((pl.col("node") == "pickaxe") & (pl.col("event") == "tool_end"))
    .group_by("trace")
    .agg(pl.len().alias("pickaxe_calls"))
)

# Join with evaluated (traces without pickaxe calls get 0)
eval_pickaxe = evaluated.join(pickaxe_counts, on="trace", how="left").with_columns(
    pl.col("pickaxe_calls").fill_null(0)
)

# Summary: mean/median/min/max pickaxe calls per outcome
print("=== Pickaxe calls by outcome ===")
display(
    eval_pickaxe.group_by("outcome")
    .agg(
        pl.len().alias("n"),
        pl.col("pickaxe_calls").mean().alias("mean"),
        pl.col("pickaxe_calls").median().alias("median"),
        pl.col("pickaxe_calls").min().alias("min"),
        pl.col("pickaxe_calls").max().alias("max"),
        pl.col("pickaxe_calls").sum().alias("total"),
    )
    .sort("outcome")
)

# Distribution: count of traces by (pickaxe_calls, outcome)
print("\n=== Distribution: traces per pickaxe call count x outcome ===")
display(
    eval_pickaxe.group_by(["pickaxe_calls", "outcome"])
    .len()
    .sort(["pickaxe_calls", "outcome"])
    .pivot(on="outcome", index="pickaxe_calls", values="len")
    .fill_null(0)
    .sort("pickaxe_calls")
)

=== Pickaxe calls by outcome ===


outcome,n,mean,median,min,max,total
str,u32,f64,f64,u32,u32,u32
"""correct""",1617,0.176871,0.0,0,8,286
"""empty""",93,0.27957,0.0,0,11,26
"""error""",39,0.74359,0.0,0,7,29
"""incorrect""",519,0.348748,0.0,0,9,181



=== Distribution: traces per pickaxe call count x outcome ===


pickaxe_calls,correct,empty,error,incorrect
u32,u32,u32,u32,u32
0,1475,84,26,433
1,78,5,6,40
2,30,1,4,23
3,14,0,1,8
4,8,2,0,10
…,…,…,…,…
6,4,0,0,2
7,2,0,1,0
8,2,0,0,0
